<a href="https://colab.research.google.com/github/Prudhvilakshman1112/GEN-AI/blob/main/EXP_5_Fine_tune_GPT_2_or_GPT_Neo_on_a_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cell 1: Setup and Dataset Loading
In this cell, we import the Hugging Face Trainer API and the datasets library. We load a "tiny" slice of the Amazon Polarity dataset (200 examples). This is ideal for a lab environment or a quick proof-of-concept to ensure the pipeline works without needing hours of training time.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (GPT2LMHeadModel, GPT2Tokenizer, Trainer,
                         TrainingArguments, DataCollatorForLanguageModeling)

# Configuration and Data Loading
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load a small sample of 200 reviews
dataset = load_dataset("amazon_polarity", split="train[:200]")
print(f"Dataset loaded with {len(dataset)} examples")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

...

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

Dataset loaded with 200 examples


#Cell 2: Tokenization and Data Preprocessing
Before the model can learn, we must convert raw text into a format it understands. We combine the review "title" and "content" into a single string. We then use the GPT-2 tokenizer to truncate or pad these strings to a uniform length of 256 tokens. Finally, we split the data into Training (90%) and Test (10%) sets.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# Set padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def tokenize_function(examples):
    # Combine title and content for the model to learn context
    texts = [t + " " + c for t, c in zip(examples['title'], examples['content'])]
    return tokenizer(texts, truncation=True, max_length=256, padding="max_length")

# Map the function over the dataset and split
tokenized = dataset.map(tokenize_function, batched=True,
                        remove_columns=dataset.column_names).train_test_split(0.1)

# Data Collator handles batching for Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Tokenization and splitting completed.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenization and splitting completed.


#Cell 3: Configuring the Trainer and Fine-Tuning
This cell defines the TrainingArguments, which specify the "rules" of the training (learning rate, number of epochs, and batch size). We then initialize the Trainer—a high-level feature that handles the complex training loop for us. Running trainer.train() starts the actual learning process.

In [ ]:
print("Starting fine-tuning of GPT-2 on product reviews...")

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator
)

# Start training
trainer.train()

# Save the model locally
trainer.save_model("./gpt2-finetuned")
print("Fine-tuning completed and model saved!")

Starting fine-tuning of GPT-2 on product reviews...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.616122
2,No log,3.610129
3,No log,3.611205


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#Cell 4: Testing the Fine-Tuned Model
To verify that the model has learned the "language" of reviews, we provide it with a prompt related to electronics. We use sampling techniques like top_p and temperature to ensure the generated text is creative and doesn't get stuck in repetitive loops.

In [ ]:
print("\n=== Generated Product Reviews ===")

prompt = "This wireless earbuds are"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate text using the fine-tuned model
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Prompt: {prompt}")
print(f"Generated Result:\n{generated}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



=== Generated Product Reviews ===
Prompt: This wireless earbuds are
Generated Result:
This wireless earbuds are so good and you can't beat the quality. The best part is they don`t cost much for anything other than a rechargeable battery - this one comes in four colors! One of these I'm using as an additional to my home theater set ups, another has been sent off (which works fine), and yet still others have no issues whatsoever with their audio output at all.. All that said
